# 第二部分：得力顾问 (系统提示词的力量)
### *Alex 的转型：从苦力程序员到 AI 架构师*

在 **Notebook 1** 中，我们见证了硬编码逻辑的“脆性”。当 `Gross_Amount` 变成 `total_price` 时，整个系统就瘫痪了。

今天，Alex 引入了 **LLM 顾问**。他不再尝试预测未来的每一种数据格式，而是写了一段“业务说明书”作为系统提示词。在这个笔记本中，我们将学习：
1. **系统提示词的艺术**：如何将死板的代码逻辑翻译成 AI 能理解的业务指令。
2. **元数据驱动推理**：让 LLM 观察新数据的表头 (Schema)，并自动“桥接”差异。
3. **观察“纯推理”的局限**：LLM 懂了，但它还没动起手来。

In [2]:
import pandas as pd
import os
# 修改这里：使用最新的导入路径
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage 

# 1. 读取我们在 Notebook 1 中生成的“崩溃数据”
modern_file = "enterprise_data/modern_marketing.csv"
if os.path.exists(modern_file):
    df_modern = pd.read_csv(modern_file)
    print("成功加载市场部新数据，准备进行 AI 诊断...")
else:
    print("错误：找不到数据文件。请确保已经运行了 Notebook 1 生成数据。")

成功加载市场部新数据，准备进行 AI 诊断...


## 1. 编写“业务说明书” (System Prompt)
Alex 将老代码的硬编码逻辑提取出来，写成了这段核心指令。这就是未来“技能 (Skill)”的雏形。

In [3]:
# 定义系统背景：不仅是翻译，还要注入企业标准
SYSTEM_PROMPT = """
你是一位 GlobalCorp 的资深数据架构专家。

我们的【旧系统可视化宏】非常死板，它只能识别以下特定格式：
- 必须有 'Region' 列用于地区分组。
- 必须有 'Quantity' 和 'Unit_Price' 列用于计算收入。
- 必须有 'Product_Category' 用于产品构成分析。

你的任务是：
1. 观察用户上传的新数据结构。
2. 找出新旧列名之间的映射关系（例如：'market' 其实就是 'Region'）。
3. 识别并修复潜在的数据格式问题（例如：带 $ 的字符串需要转为数字）。
4. 给出具体的 Python 修复建议，让新数据能够顺利通过旧系统的验证。
"""

## 2. 顾问上线：进行“无代码”诊断
我们只给 AI 看数据的前几行（Schema），看它能否理解如何修复问题。

In [4]:
# 设置你的 API KEY (请确保环境中有该变量，或者在此手动设置)
# os.environ["OPENAI_API_KEY"] = "sk-xxxx..."
from langchain.chat_models import init_chat_model


try:
    llm = model = init_chat_model(
    model="agnes-2.0-flash",
    model_provider="openai",
    base_url=os.getenv("AGNES_BASE_URL"),
    api_key=os.getenv("AGNES_API_KEY"),
)


    def run_ai_consultant(df):
        # 提取元数据（列名和前两行示例）
        metadata = df.head(2).to_string()
        
        user_question = f"""
        我的旧报表宏崩溃了。这是新数据的样板：
        {metadata}
        
        请帮我分析差异，并告诉我需要写什么样的 Python 代码来适配旧系统？
        """
        
        messages = [
            SystemMessage(content=SYSTEM_PROMPT),
            HumanMessage(content=user_question)
        ]
        
        response = llm.invoke(messages)
        return response.content

    report = run_ai_consultant(df_modern)
    print("--- AI 顾问的诊断报告 ---\n")
    print(report)
except Exception as e:
    print(f"连接 AI 失败，请检查 API Key 设置。错误详情: {e}")

--- AI 顾问的诊断报告 ---

你好，我是 Agnes-2.0-Flash。作为 GlobalCorp 的数据架构专家，我仔细审查了你提供的新数据结构与旧系统宏要求的差异。

以下是我的分析及修复方案：

### 1. 差异分析与映射关系

| 旧系统要求 (Required) | 新数据列名 (New Data) | 状态 | 备注 |
| :--- | :--- | :--- | :--- |
| `Region` | `market` | **需重命名** | 语义一致 (US/EMEA) |
| `Quantity` | `qty` | **需重命名** | 语义一致 |
| `Unit_Price` | `price_per_unit` | **需重命名** | 数据格式需清洗 (含 `$` 和 `,`) |
| `Product_Category` | `prod_cat` | **需重命名** | 语义一致 |
| *(无要求)* | `tx_id` | **需丢弃** | 旧系统不需要此 ID |
| *(无要求)* | `rebate` | **需丢弃** | 旧系统不需要此折扣字段 |

### 2. 潜在数据格式问题

1.  **货币格式干扰**：`price_per_unit` 列包含美元符号 `$` 和千位分隔符 `,`（如 `'$1,250.00'`）。这会导致后续计算将数字视为字符串，从而崩溃。必须转换为浮点数。
2.  **列名不匹配**：旧宏硬编码了特定列名，直接传入新 DataFrame 会报 `KeyError`。

### 3. Python 修复建议代码

请使用以下代码段清洗数据，使其通过旧系统的验证：

```python
import pandas as pd

def adapt_data_for_legacy_macro(df):
    """
    将新结构数据转换为旧系统宏可接受的格式。
    """
    # 1. 创建副本以避免修改原始数据
    cleaned_df = df.copy()
    
    # 2. 处理 Unit_Price 数据清洗
    # 移除 '$' 和 ','，然后转换为浮点数
    cleaned_df['Unit_Price'] = (
     

## 3. 深度复盘：Alex 的新烦恼

通过上面的输出，你会发现 AI 非常强大：
- 它一眼看出 `qty` 就是 `Quantity`。
- 它建议用 `.str.replace('$', '')` 来修复价格列。
- 它甚至理解 `market` 应该映射为 `Region`。

### 但是，为什么这还不是“自动化”？
观察 AI 的反馈。虽然它给出了完美的修复建议，但 Alex 依然需要：
1. **阅读** AI 的回复。
2. **复制** AI 给出的 Python 代码块。
3. **粘贴** 到自己的脚本里运行。

这依然是 **“人机协作”**，而不是 **“智能体自主 (Agentic Action)”**。

### 核心痛点
这就是**单纯使用系统提示词 (System Prompt)** 的天花板：它能提供“大脑”，但它没有“手”。

**在接下来的 Notebook 3 中，我们将为 Alex 的 AI 顾问安装一只“手”——Python REPL 工具。我们将让 AI 直接编写并运行修复代码！**